# Single-stage optimization checks

Interactive equivalents of `test_optimize.py`. MMFF and UFF use RDKit locally, so no optional executable is required.

In [ ]:
from pathlib import Path
import sys
import numpy as np

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'src').is_dir():
    project_root = project_root.parent
if not (project_root / 'src' / 'ensemblelab').is_dir():
    raise RuntimeError('Open this notebook from within the ensemblelab repository.')
sys.path.insert(0, str(project_root / 'src'))

from ensemblelab import generate, optimize


## MMFF preserves the source ensemble

Optimization returns a distinct ensemble with kcal/mol energies and synchronized RDKit/ASE coordinates.

In [ ]:
ensemble = generate('CCO', n_confs=2)
original_positions = [conformer.atoms.positions.copy() for conformer in ensemble.conformers]
optimized = optimize(ensemble, method='MMFF', max_steps=100)

assert optimized is not ensemble
assert all(conformer.energy is None for conformer in ensemble.conformers)
assert all(conformer.energy is not None for conformer in optimized.conformers)
assert all(conformer.energy_unit == 'kcal/mol' for conformer in optimized.conformers)
assert all(conformer.optimization_method == 'MMFF' for conformer in optimized.conformers)
assert optimized.metadata['energy_status'] == 'computed'
assert optimized.metadata['optimization_history'][-1]['method'] == 'MMFF'

for before, original, result in zip(original_positions, ensemble.conformers, optimized.conformers, strict=True):
    np.testing.assert_allclose(original.atoms.positions, before)
    np.testing.assert_allclose(result.atoms.positions, optimized.rdkit_conformer(result.id).GetPositions())

print('MMFF contract passed.')
[(conformer.id, conformer.energy) for conformer in optimized.conformers]


## UFF alias and solvent validation

The string API accepts lowercase backend aliases. Solvation is currently limited to the GFN2-xTB backend.

In [ ]:
uff_ensemble = optimize(ensemble, method='uff', max_steps=100)
assert uff_ensemble.conformers[0].optimization_method == 'UFF'

try:
    optimize(ensemble, method='MMFF', solvent='water')
except ValueError as error:
    assert 'GFN2-xTB' in str(error)
    print(f'Expected validation error: {error}')
else:
    raise AssertionError('MMFF with solvent should raise ValueError')
